# Bài lab PySpark RDD - Lời giải

Notebook này dùng **RDD** theo đúng yêu cầu đề bài.  
Đặt các file `movies.txt`, `ratings_1.txt`, `ratings_2.txt`, `users.txt`, `occupation.txt` cùng thư mục với notebook rồi chạy lần lượt từng cell.

**Dữ liệu:** 

1. movies.txt
- **Schema**: MovieID, Title, Genres

2. ratings_1.txt, ratings_2.txt
- **Schema**: UserID, MovieID, Rating, Timestamp

3. users.txt
- **Schema**: UserID, Gender, Age, Occupation, Zip-code

4. occupation.txt 
- **Schema**: ID, Occupation

**Lưu ý:** Thực hiện các bài tập dưới đây sử dụng RDD.

In [9]:
import os
import sys
import subprocess
from datetime import datetime

JAVA_HOME = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
SPARK_HOME = r"C:\spark\spark-4.1.2-bin-hadoop3"

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["SPARK_HOME"] = SPARK_HOME

os.environ["PATH"] = (
    os.path.join(JAVA_HOME, "bin") + ";" +
    os.path.join(SPARK_HOME, "bin") + ";" +
    os.environ["PATH"]
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python:", sys.executable)
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("SPARK_HOME:", os.environ["SPARK_HOME"])
print("spark-submit exists:", os.path.exists(os.path.join(SPARK_HOME, "bin", "spark-submit.cmd")))

java_test = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(java_test.stderr)

Python: c:\Users\Admin\AppData\Local\Programs\python.exe
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot
SPARK_HOME: C:\spark\spark-4.1.2-bin-hadoop3
spark-submit exists: True
openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Temurin-17.0.19+10 (build 17.0.19+10)
OpenJDK 64-Bit Server VM Temurin-17.0.19+10 (build 17.0.19+10, mixed mode, sharing)



In [10]:


import os
from datetime import datetime
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MovieRatingRDDLab") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

def resolve_path(filename):
    """
    Ưu tiên đọc file trong cùng thư mục notebook.
    Nếu chạy trên môi trường khác, có thể sửa BASE_DIR bên dưới.
    """
    candidates = [
        os.path.join(os.getcwd(), filename),
        os.path.join("/mnt/data", filename),  # dùng khi chạy trong sandbox/Colab-like env
        filename
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return filename

MOVIES_PATH = resolve_path("movies.txt")
RATINGS_1_PATH = resolve_path("ratings_1.txt")
RATINGS_2_PATH = resolve_path("ratings_2.txt")
USERS_PATH = resolve_path("users.txt")
OCCUPATION_PATH = resolve_path("occupation.txt")

print("Movies:", MOVIES_PATH)
print("Ratings 1:", RATINGS_1_PATH)
print("Ratings 2:", RATINGS_2_PATH)
print("Users:", USERS_PATH)
print("Occupation:", OCCUPATION_PATH)

Movies: d:\study\Big data\Lap 3\movies.txt
Ratings 1: d:\study\Big data\Lap 3\ratings_1.txt
Ratings 2: d:\study\Big data\Lap 3\ratings_2.txt
Users: d:\study\Big data\Lap 3\users.txt
Occupation: d:\study\Big data\Lap 3\occupation.txt


In [11]:
# Đọc và parse dữ liệu bằng RDD

def parse_movie(line):
    # Schema: MovieID, Title, Genres
    movie_id, title, genres = line.strip().split(",", 2)
    return int(movie_id), title, genres.split("|")

def parse_rating(line):
    # Schema: UserID, MovieID, Rating, Timestamp
    user_id, movie_id, rating, timestamp = line.strip().split(",")
    return int(user_id), int(movie_id), float(rating), int(timestamp)

def parse_user(line):
    # Schema: UserID, Gender, Age, Occupation, Zip-code
    user_id, gender, age, occupation_id, zip_code = line.strip().split(",")
    return int(user_id), gender, int(age), int(occupation_id), zip_code

def parse_occupation(line):
    # Schema: ID, Occupation
    occupation_id, occupation = line.strip().split(",", 1)
    return int(occupation_id), occupation

movies_rdd = sc.textFile(MOVIES_PATH).filter(lambda x: x.strip() != "").map(parse_movie)
ratings_rdd = sc.textFile(RATINGS_1_PATH) \
    .union(sc.textFile(RATINGS_2_PATH)) \
    .filter(lambda x: x.strip() != "") \
    .map(parse_rating)
users_rdd = sc.textFile(USERS_PATH).filter(lambda x: x.strip() != "").map(parse_user)
occupation_rdd = sc.textFile(OCCUPATION_PATH).filter(lambda x: x.strip() != "").map(parse_occupation)

movie_titles_rdd = movies_rdd.map(lambda x: (x[0], x[1]))

print("Số phim:", movies_rdd.count())
print("Số rating:", ratings_rdd.count())
print("Số user:", users_rdd.count())
print("Số occupation:", occupation_rdd.count())

Số phim: 50
Số rating: 184
Số user: 50
Số occupation: 15


**Bài 1**: Tính điểm trung bình và tổng số lượt đánh giá cho mỗi phim
* **Mục tiêu**:
  * Tính điểm trung bình của mỗi phim.
  * Đếm tổng số lượt đánh giá.
  * Tìm phim có điểm trung bình cao nhất (chỉ xét những phim có ít nhất 50 lượt đánh giá).
* **Giải pháp**:
  * Bước 1: Đọc file movies.txt và tạo một map (MovieID → Title).
  * Bước 2: Đọc file ratings_1.txt và ratings_2.txt, map MovieID → (Rating, 1).
  * Bước 3: Reduce để tính tổng điểm và số lượt đánh giá.
  * Bước 4: Tính điểm trung bình, lọc ra phim có ít nhất 5 lượt đánh giá.
  * Bước 5: Tìm phim có điểm trung bình cao nhất.

## Bài 1: Tính điểm trung bình và tổng số lượt đánh giá cho mỗi phim

**Ghi chú:** Trong đề có chỗ ghi “ít nhất 50 lượt đánh giá”, nhưng phần giải pháp lại ghi “ít nhất 5 lượt đánh giá”.  
Dataset lab này không có phim nào đạt 50 lượt, nên cell bên dưới để `MIN_REVIEWS = 5`. Nếu giảng viên yêu cầu đúng 50 thì đổi lại thành `50`.

In [12]:
MIN_REVIEWS = 5

# MovieID -> (Rating, 1)
movie_rating_pairs = ratings_rdd.map(lambda x: (x[1], (x[2], 1)))

# MovieID -> (total_rating, total_count)
movie_rating_totals = movie_rating_pairs.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)

# MovieID -> (avg_rating, total_count)
movie_rating_avg = movie_rating_totals.mapValues(
    lambda x: (x[0] / x[1], x[1])
)

# Join với tên phim để dễ đọc
movie_rating_result = movie_rating_avg.join(movie_titles_rdd) \
    .map(lambda x: (x[0], x[1][1], round(x[1][0][0], 3), x[1][0][1])) \
    .sortBy(lambda x: (-x[2], -x[3], x[1]))

print("Điểm trung bình và tổng số lượt đánh giá cho mỗi phim:")
for movie_id, title, avg_rating, count in movie_rating_result.collect():
    print(f"{movie_id} | {title} | avg={avg_rating} | count={count}")

highest_rated_movie = movie_rating_avg \
    .filter(lambda x: x[1][1] >= MIN_REVIEWS) \
    .join(movie_titles_rdd) \
    .map(lambda x: (x[1][0][0], x[1][0][1], x[0], x[1][1])) \
    .takeOrdered(1, key=lambda x: (-x[0], -x[1], x[3]))

print(f"\nPhim có điểm trung bình cao nhất với ít nhất {MIN_REVIEWS} lượt đánh giá:")
for avg_rating, count, movie_id, title in highest_rated_movie:
    print(f"{movie_id} | {title} | avg={round(avg_rating, 3)} | count={count}")

Điểm trung bình và tổng số lượt đánh giá cho mỗi phim:
1015 | Sunset Boulevard (1950) | avg=4.357 | count=7
1025 | The Terminator (1984) | avg=4.056 | count=18
1013 | The Godfather: Part II (1974) | avg=4.0 | count=17
1012 | Psycho (1960) | avg=4.0 | count=2
1043 | No Country for Old Men (2007) | avg=3.889 | count=18
1037 | The Lord of the Rings: The Fellowship of the Ring (2001) | avg=3.889 | count=18
1047 | The Social Network (2010) | avg=3.857 | count=7
1039 | The Lord of the Rings: The Return of the King (2003) | avg=3.818 | count=11
1020 | E.T. the Extra-Terrestrial (1982) | avg=3.667 | count=18
1040 | Gladiator (2000) | avg=3.611 | count=18
1028 | Fight Club (1999) | avg=3.5 | count=7
1050 | Mad Max: Fury Road (2015) | avg=3.472 | count=18
1010 | Lawrence of Arabia (1962) | avg=3.444 | count=18
1030 | The Silence of the Lambs (1991) | avg=3.143 | count=7

Phim có điểm trung bình cao nhất với ít nhất 5 lượt đánh giá:
1015 | Sunset Boulevard (1950) | avg=4.357 | count=7


Bài 2: Phân tích đánh giá theo thể loại
* Mục tiêu:
  * Tính điểm trung bình của từng thể loại phim.
* Giải pháp
  * Bước 1: Tạo map (MovieID → List of Genres).
  * Bước 2: Map từ MovieID → Rating → (Genre, Rating).
  * Bước 3: Tính trung bình điểm đánh giá cho từng thể loại.

## Bài 2: Phân tích đánh giá theo thể loại

Tính điểm trung bình của từng thể loại phim.

In [13]:
# MovieID -> [Genres]
movie_genres_rdd = movies_rdd.map(lambda x: (x[0], x[2]))

# MovieID -> Rating
rating_by_movie_rdd = ratings_rdd.map(lambda x: (x[1], x[2]))

# Join rating với genres, sau đó explode ra Genre -> (Rating, 1)
genre_rating_pairs = rating_by_movie_rdd.join(movie_genres_rdd) \
    .flatMap(lambda x: [(genre, (x[1][0], 1)) for genre in x[1][1]])

genre_rating_avg = genre_rating_pairs.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).mapValues(
    lambda x: (x[0] / x[1], x[1])
).sortBy(
    lambda x: (-x[1][0], x[0])
)

print("Điểm trung bình theo thể loại:")
for genre, (avg_rating, count) in genre_rating_avg.collect():
    print(f"{genre} | avg={round(avg_rating, 3)} | count={count}")

Điểm trung bình theo thể loại:
Film-Noir | avg=4.357 | count=7
Horror | avg=4.0 | count=2
Mystery | avg=4.0 | count=2
Fantasy | avg=3.862 | count=29
Crime | avg=3.81 | count=42
Drama | avg=3.758 | count=128
Sci-Fi | avg=3.731 | count=54
Action | avg=3.713 | count=54
Thriller | avg=3.704 | count=27
Family | avg=3.667 | count=18
Adventure | avg=3.633 | count=83
Biography | avg=3.56 | count=25


Bài 3: Phân tích đánh giá theo giới tính
* Mục tiêu:
  * Tính điểm trung bình của mỗi phim theo giới tính.
* Giải pháp
  * Bước 1: Tạo map (UserID → Gender).
  * Bước 2: Join với ratings để thêm thông tin giới tính.
  * Bước 3: Tính trung bình rating cho mỗi phim theo từng giới tính.

## Bài 3: Phân tích đánh giá theo giới tính

Tính điểm trung bình của mỗi phim theo giới tính.

In [14]:
# UserID -> Gender
user_gender_rdd = users_rdd.map(lambda x: (x[0], x[1]))

# UserID -> (MovieID, Rating)
rating_by_user_rdd = ratings_rdd.map(lambda x: (x[0], (x[1], x[2])))

# Join để có: UserID -> ((MovieID, Rating), Gender)
# Sau đó map: (MovieID, Gender) -> (Rating, 1)
movie_gender_rating_pairs = rating_by_user_rdd.join(user_gender_rdd) \
    .map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1)))

movie_gender_avg = movie_gender_rating_pairs.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).mapValues(
    lambda x: (x[0] / x[1], x[1])
)

# Join tên phim
movie_gender_result = movie_gender_avg \
    .map(lambda x: (x[0][0], (x[0][1], x[1][0], x[1][1]))) \
    .join(movie_titles_rdd) \
    .map(lambda x: (x[1][1], x[1][0][0], round(x[1][0][1], 3), x[1][0][2])) \
    .sortBy(lambda x: (x[0], x[1]))

print("Điểm trung bình của mỗi phim theo giới tính:")
for title, gender, avg_rating, count in movie_gender_result.collect():
    print(f"{title} | Gender={gender} | avg={avg_rating} | count={count}")

Điểm trung bình của mỗi phim theo giới tính:
E.T. the Extra-Terrestrial (1982) | Gender=F | avg=3.55 | count=10
E.T. the Extra-Terrestrial (1982) | Gender=M | avg=3.812 | count=8
Fight Club (1999) | Gender=F | avg=3.5 | count=3
Fight Club (1999) | Gender=M | avg=3.5 | count=4
Gladiator (2000) | Gender=F | avg=3.643 | count=7
Gladiator (2000) | Gender=M | avg=3.591 | count=11
Lawrence of Arabia (1962) | Gender=F | avg=3.312 | count=8
Lawrence of Arabia (1962) | Gender=M | avg=3.55 | count=10
Mad Max: Fury Road (2015) | Gender=F | avg=3.321 | count=14
Mad Max: Fury Road (2015) | Gender=M | avg=4.0 | count=4
No Country for Old Men (2007) | Gender=F | avg=3.833 | count=6
No Country for Old Men (2007) | Gender=M | avg=3.917 | count=12
Psycho (1960) | Gender=F | avg=4.0 | count=2
Sunset Boulevard (1950) | Gender=F | avg=4.5 | count=1
Sunset Boulevard (1950) | Gender=M | avg=4.333 | count=6
The Godfather: Part II (1974) | Gender=F | avg=3.938 | count=8
The Godfather: Part II (1974) | Gender=M

Bài 4: Phân tích đánh giá theo nhóm tuổi
* Mục tiêu:
  * Phân loại người dùng theo nhóm tuổi và tính điểm trung bình của mỗi phim theo từng nhóm.
* Giải pháp
  * Bước 1: Tạo map (UserID → Age Group).
  * Bước 2: Join với ratings để thêm nhóm tuổi.
  * Bước 3: Tính trung bình điểm đánh giá theo nhóm tuổi.

## Bài 4: Phân tích đánh giá theo nhóm tuổi

Phân loại người dùng theo nhóm tuổi và tính điểm trung bình của mỗi phim theo từng nhóm.

In [15]:
def get_age_group(age):
    if age < 18:
        return "<18"
    elif age <= 24:
        return "18-24"
    elif age <= 34:
        return "25-34"
    elif age <= 44:
        return "35-44"
    elif age <= 49:
        return "45-49"
    elif age <= 55:
        return "50-55"
    else:
        return "56+"

# UserID -> AgeGroup
user_age_group_rdd = users_rdd.map(lambda x: (x[0], get_age_group(x[2])))

# UserID -> (MovieID, Rating)
rating_by_user_rdd = ratings_rdd.map(lambda x: (x[0], (x[1], x[2])))

# (MovieID, AgeGroup) -> (Rating, 1)
movie_age_rating_pairs = rating_by_user_rdd.join(user_age_group_rdd) \
    .map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1)))

movie_age_avg = movie_age_rating_pairs.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).mapValues(
    lambda x: (x[0] / x[1], x[1])
)

# Join tên phim
movie_age_result = movie_age_avg \
    .map(lambda x: (x[0][0], (x[0][1], x[1][0], x[1][1]))) \
    .join(movie_titles_rdd) \
    .map(lambda x: (x[1][1], x[1][0][0], round(x[1][0][1], 3), x[1][0][2])) \
    .sortBy(lambda x: (x[0], x[1]))

print("Điểm trung bình của mỗi phim theo nhóm tuổi:")
for title, age_group, avg_rating, count in movie_age_result.collect():
    print(f"{title} | AgeGroup={age_group} | avg={avg_rating} | count={count}")

Điểm trung bình của mỗi phim theo nhóm tuổi:
E.T. the Extra-Terrestrial (1982) | AgeGroup=18-24 | avg=3.5 | count=2
E.T. the Extra-Terrestrial (1982) | AgeGroup=25-34 | avg=3.583 | count=6
E.T. the Extra-Terrestrial (1982) | AgeGroup=35-44 | avg=3.812 | count=8
E.T. the Extra-Terrestrial (1982) | AgeGroup=45-49 | avg=4.0 | count=1
E.T. the Extra-Terrestrial (1982) | AgeGroup=56+ | avg=3.0 | count=1
Fight Club (1999) | AgeGroup=25-34 | avg=3.5 | count=3
Fight Club (1999) | AgeGroup=35-44 | avg=3.5 | count=2
Fight Club (1999) | AgeGroup=50-55 | avg=3.5 | count=2
Gladiator (2000) | AgeGroup=18-24 | avg=3.5 | count=1
Gladiator (2000) | AgeGroup=25-34 | avg=3.417 | count=6
Gladiator (2000) | AgeGroup=35-44 | avg=3.75 | count=6
Gladiator (2000) | AgeGroup=45-49 | avg=4.0 | count=2
Gladiator (2000) | AgeGroup=50-55 | avg=3.75 | count=2
Gladiator (2000) | AgeGroup=56+ | avg=3.0 | count=1
Lawrence of Arabia (1962) | AgeGroup=25-34 | avg=3.6 | count=5
Lawrence of Arabia (1962) | AgeGroup=35-44 |

Bài 5: Phân Tích Đánh Giá Theo Occupation (Nghề nghiệp) Của Người Dùng
* Mục tiêu:
  * Tính trung bình rating và tổng số lượt đánh giá cho từng Occupation.
* Giải pháp:
  * Tạo dictionary từ users.txt với mapping UserID → Occupation.
  * Với mỗi rating, gán thông tin Occupation theo UserID.
  * Phát hành cặp key-value với key là Occupation và value là (rating, 1).
  * Reduce để tính tổng điểm và số lượt cho mỗi Occupation, sau đó tính trung bình rating.

## Bài 5: Phân tích đánh giá theo Occupation của người dùng

Tính trung bình rating và tổng số lượt đánh giá cho từng nghề nghiệp.

In [ ]:

occupation_map = occupation_rdd.collectAsMap()
broadcast_occupation_map = sc.broadcast(occupation_map)


user_occupation_rdd = users_rdd.map(
    lambda x: (x[0], broadcast_occupation_map.value.get(x[3], "Unknown"))
)


rating_by_user_only_rdd = ratings_rdd.map(lambda x: (x[0], x[2]))


occupation_rating_pairs = rating_by_user_only_rdd.join(user_occupation_rdd) \
    .map(lambda x: (x[1][1], (x[1][0], 1)))

occupation_rating_avg = occupation_rating_pairs.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).mapValues(
    lambda x: (x[0] / x[1], x[1])
).sortBy(
    lambda x: (-x[1][0], x[0])
)

print("Trung bình rating và tổng số lượt đánh giá theo nghề nghiệp:")
for occupation, (avg_rating, count) in occupation_rating_avg.collect():
    print(f"{occupation} | avg={round(avg_rating, 3)} | count={count}")

Trung bình rating và tổng số lượt đánh giá theo nghề nghiệp:
Programmer | avg=4.25 | count=10
Designer | avg=4.0 | count=13
Student | avg=4.0 | count=8
Nurse | avg=3.864 | count=11
Consultant | avg=3.857 | count=14
Journalist | avg=3.853 | count=17
Artist | avg=3.727 | count=11
Teacher | avg=3.7 | count=5
Doctor | avg=3.69 | count=21
Lawyer | avg=3.647 | count=17
Salesperson | avg=3.647 | count=17
Accountant | avg=3.583 | count=6
Engineer | avg=3.556 | count=18
Manager | avg=3.469 | count=16


Bài 6: Phân Tích Đánh Giá Theo Thời Gian
* Mục tiêu:
  * Tính tổng số lượt đánh giá và điểm trung bình cho mỗi năm.
* Giải pháp:
  * Đọc dữ liệu ratings (từ cả ratings_1.txt và ratings_2.txt).
  * Sử dụng hàm trợ giúp để chuyển đổi Timestamp (dạng Unix) thành năm (Year).
  * Với mỗi dòng ratings, phát hành cặp key-value với key là năm và value là (rating, 1).
  * Reduce để tính tổng điểm và số lượt cho mỗi năm, sau đó tính trung bình.

## Bài 6: Phân tích đánh giá theo thời gian

Tính tổng số lượt đánh giá và điểm trung bình cho mỗi năm.

In [17]:
def timestamp_to_year(timestamp):
    return datetime.utcfromtimestamp(timestamp).year

# Year -> (Rating, 1)
year_rating_pairs = ratings_rdd.map(
    lambda x: (timestamp_to_year(x[3]), (x[2], 1))
)

year_rating_avg = year_rating_pairs.reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).mapValues(
    lambda x: (x[0] / x[1], x[1])
).sortByKey()

print("Tổng số lượt đánh giá và điểm trung bình theo năm:")
for year, (avg_rating, count) in year_rating_avg.collect():
    print(f"{year} | avg={round(avg_rating, 3)} | count={count}")

Tổng số lượt đánh giá và điểm trung bình theo năm:
2020 | avg=3.753 | count=184


In [18]:
spark.stop()